In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv('data/mnist_train.csv')

In [3]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df = np.array(df)
r, c = df.shape
np.random.shuffle(df)

In [7]:
df = df.T

In [8]:
y_data = df[0]
y_data = y_data.reshape(1, 60000)

x_data = df[1:].T

In [9]:
r, c = c, r

In [10]:
x_data

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(60000, 784))

In [11]:
y_data

array([[5, 1, 0, ..., 3, 2, 1]], shape=(1, 60000))

In [15]:
def init_params():
    W1 = np.random.randn(784, 8) * np.sqrt(2. / 784)
    B1 = np.random.randn(1, 8)
    W2 = np.random.randn(8, 16) * np.sqrt(2. / 8)
    B2 = np.random.randn(1, 16)
    W3 = np.random.randn(16, 16) * np.sqrt(2. / 16)
    B3 = np.random.randn(1, 16)
    W4 = np.random.randn(16, 10) * np.sqrt(2. / 16)
    B4 = np.random.randn(1, 10)
    return W1, B1, W2, B2, W3, B3, W4, B4

def create_batches(y_data, x_data):
    y_batches = y_data.reshape(60, 1000, -1)
    x_batches = x_data.reshape(60, 1000, -1)

    return y_batches, x_batches

def forward(x_batch, weights, bias):
    Z1 = x_batch @ weights[0] + bias[0]
    A1 = tanh(Z1)
    
    Z2 = A1 @ weights[1] + bias[1]
    A2 = tanh(Z2)
    
    Z3 = A2 @ weights[2] + bias[2]
    A3 = tanh(Z3)
    
    Z4 = A3 @ weights[3] + bias[3]
    y_pred = softmax(Z4)

    return y_pred, Z4, A3, Z3, A2, Z2, A1, Z1

def tanh(Z):
    return np.tanh(Z)

def softmax(Z):
    Z_max = np.max(Z, axis=1, keepdims=True)
    Z_shifted = Z - Z_max
    Z_exp = np.exp(Z_shifted)

    prob = Z_exp / np.sum(Z_exp, axis=1, keepdims=True)
    return prob

def to_one_hot(y_batch, num_classes=10):
    y_flat = y_batch.flatten().astype(int)
    y_one_hot = np.eye(num_classes)[y_flat]
    return y_one_hot

def compute_loss(y_pred, y_true):
    m = y_pred.shape[0]
    epsilon = 1e-15
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    loss_matrix = y_true * np.log(y_pred_clipped)
    cost = -np.sum(loss_matrix) / m
    return cost


Блок для SGD + momentum

In [ ]:
def init_momentum_state(weights, bias):
    """Инициализация скоростей (v) нулями для каждого слоя"""
    v_w = [np.zeros_like(w) for w in weights]
    v_b = [np.zeros_like(b) for b in bias]
    return v_w, v_b

def update_sgd_momentum(weights, bias, d_weights, d_bias, v_w, v_b, learning_rate=0.01, beta=0.9):
    """
    Шаг оптимизации SGD + Momentum.
    beta (обычно 0.9) - коэффициент сохранения импульса.
    """
    num_layers = len(weights)
    
    for i in range(num_layers):
        # Обновляем "скорость" для весов и смещений
        v_w[i] = beta * v_w[i] + (1 - beta) * d_weights[i]
        v_b[i] = beta * v_b[i] + (1 - beta) * d_bias[i]
        
        # Обновляем сами параметры
        weights[i] = weights[i] - learning_rate * v_w[i]
        bias[i] = bias[i] - learning_rate * v_b[i]
        
    return weights, bias, v_w, v_b

In [13]:
y, x = create_batches(y_data, x_data)
y

array([[[5],
        [1],
        [0],
        ...,
        [3],
        [0],
        [5]],

       [[7],
        [7],
        [0],
        ...,
        [8],
        [5],
        [9]],

       [[9],
        [3],
        [8],
        ...,
        [9],
        [7],
        [7]],

       ...,

       [[7],
        [8],
        [0],
        ...,
        [7],
        [9],
        [1]],

       [[1],
        [4],
        [1],
        ...,
        [7],
        [3],
        [3]],

       [[4],
        [3],
        [4],
        ...,
        [3],
        [2],
        [1]]], shape=(60, 1000, 1))